In [13]:
import pandas as pd
from mistralai import Mistral

mistral = Mistral(api_key="UTILISE_TA_CLE")

In [14]:
from datasets import load_dataset

In [15]:
ds = load_dataset('arch-be/brabant-xvii', name="doc_by_doc")

In [16]:
from datasets import Dataset

df = pd.concat([
    ds['train'].to_pandas(), 
    ds['test'].to_pandas(), 
    ds['valid'].to_pandas()
])
df

,project,file_id,text
0,pardons,299.txt,kaerle etc doen te wetene allen iegewoirdich e...
1,pardons,228.txt,remissie voer\ndiericken gheerits\nkaerle etc....
2,pardons,118.txt,philips etc. allen etc. saluit alsoe als tonse...
3,pardons,126.txt,philips byder gratien goidts coninck van casti...
4,pardons,373.txt,remissie van dootslage\nphilips byder gracien ...
...,...,...,...
62,pardons,012.txt,maximilian par la grace de dieu esleu empereur...
63,pardons,099.txt,remissie\nalbert ende isabel clara eugenia etc...
64,pardons,315.txt,kaerle etc. doen te wetene allen iegewoirdigen...
65,pardons,300.txt,kaerle etc. doen te wetene allen iegewoirdige ...


In [21]:
def keywords(x: str): 
    prompt = f"""
    {x}. 
    Give me 10 keywords in English to convey the gist of the facts related in the ancient text above as a json list of words. 
    For example, if the text talks about a drunk man who shot his girlfriend in the head with a bat, you should return:
    ['drunk', 'shot', 'bat', 'head', 'murder']
    """
    return mistral.chat.complete(
        model = 'open-mistral-nemo',
        messages = [
            {
                "role": "user",
                "content": prompt,
            },
        ],
        response_format = {
          "type": "json_object",
        }
    )

In [22]:
df["keywords"] = ""
df

,project,file_id,text,keywords
0,pardons,299.txt,kaerle etc doen te wetene allen iegewoirdich e...,
1,pardons,228.txt,remissie voer\ndiericken gheerits\nkaerle etc....,
2,pardons,118.txt,philips etc. allen etc. saluit alsoe als tonse...,
3,pardons,126.txt,philips byder gratien goidts coninck van casti...,
4,pardons,373.txt,remissie van dootslage\nphilips byder gracien ...,
...,...,...,...,...
62,pardons,012.txt,maximilian par la grace de dieu esleu empereur...,
63,pardons,099.txt,remissie\nalbert ende isabel clara eugenia etc...,
64,pardons,315.txt,kaerle etc. doen te wetene allen iegewoirdigen...,
65,pardons,300.txt,kaerle etc. doen te wetene allen iegewoirdige ...,


In [27]:
from tqdm import tqdm
from time import sleep

results = []

for idx, row in tqdm(df.iterrows()):
    text = row['text']
    done = False
    while not done:
        try:
            kwds = keywords(text)
            done = True
        except:
            sleep(60)
    results.append(kwds)

df['keywords'] = results
df

419it [10:48,  1.55s/it]
/tmp/ipykernel_9230/2422250428.py:17: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  df[:5]['keywords'] = results


ValueError: Length of values (419) does not match length of index (5)

In [28]:
df['keywords'] = results
df

,project,file_id,text,keywords
0,pardons,299.txt,kaerle etc doen te wetene allen iegewoirdich e...,id='4415941e19894e85969848a953ddb924' object='...
1,pardons,228.txt,remissie voer\ndiericken gheerits\nkaerle etc....,id='912d405538a548849c9f3a77fb7356fd' object='...
2,pardons,118.txt,philips etc. allen etc. saluit alsoe als tonse...,id='f47451aac6ef42e78987a10592114363' object='...
3,pardons,126.txt,philips byder gratien goidts coninck van casti...,id='c5a0f4f141ca43abab99733355bf4aef' object='...
4,pardons,373.txt,remissie van dootslage\nphilips byder gracien ...,id='f3a224e021ac4c9f824e59343baa2933' object='...
...,...,...,...,...
62,pardons,012.txt,maximilian par la grace de dieu esleu empereur...,id='71b2147fa0494dc199f7d5b0d900a986' object='...
63,pardons,099.txt,remissie\nalbert ende isabel clara eugenia etc...,id='5ea31e3a17dd4179b186ed730f960bea' object='...
64,pardons,315.txt,kaerle etc. doen te wetene allen iegewoirdigen...,id='c5d326e71dec433490a08e48a0f221bd' object='...
65,pardons,300.txt,kaerle etc. doen te wetene allen iegewoirdige ...,id='743ab1ca075b4e6ca147417ab5f332a6' object='...


In [29]:
results[0]

ChatCompletionResponse(id='4415941e19894e85969848a953ddb924', object='chat.completion', model='open-mistral-nemo', usage=UsageInfo(prompt_tokens=2165, completion_tokens=54, total_tokens=2219), created=1739540877, choices=[ChatCompletionChoice(index=0, message=AssistantMessage(content='[\n  "petition",\n  "drunkenness",\n  "fight",\n  "injury",\n  "death",\n  "absentee",\n  "remission",\n  "pardon",\n  "justice",\n  "compensation"\n]', tool_calls=None, prefix=False, role='assistant'), finish_reason='stop')])

In [31]:
df["keywords"] = df["keywords"].

In [40]:
rx = list(map(lambda x: x.choices[0].message.content, results))

In [42]:
df["keywords"] = rx

In [46]:
list(map(lambda x: eval(x), rx))[0]

IndentationError: unexpected indent (<string>, line 205)

In [49]:
df.to_csv("keywords_xls.csv", sep=";")

In [52]:
df['keywords'].map(lambda x: x['keywords'] if isinstance(x, dict) else x)

0     [\n  "petition",\n  "drunkenness",\n  "fight",...
1     [\n  "infidelity",\n  "abandonment",\n  "betra...
2     {\n  "keywords": [\n    "apology",\n    "forgi...
3     {\n  "keywords": [\n    "murder",\n    "self-d...
4     {\n  "keywords": [\n    "assault",\n    "death...
                            ...                        
62    [\n  "theft",\n  "apology",\n  "punishment",\n...
63    [\n  "drunkenness",\n  "argument",\n  "acciden...
64    [\n  "fight",\n  "stabbing",\n  "injury",\n  "...
65    {\n  "keywords": [\n    "drunkenness",\n    "f...
66    {\n  "keywords": [\n    "drunkenness",\n    "f...
Name: keywords, Length: 419, dtype: object

In [55]:
df['keywords'][2]

2    {\n  "keywords": [\n    "apology",\n    "forgi...
2    [\n  "remissie", "dootslag", "philippe", "taye...
2    [\n  "nocturnal",\n  "disturbance",\n  "demand...
Name: keywords, dtype: object